# Embeddings

Loads the same preprocessed GoEmotions dataset as Project 1 (via `projects/src/data.py`).

In [5]:
# Setup: make the course code (part1) and our shared code (projects/src) importable.
# Run from projects/project2 (VS Code / JupyterLab do this by default).
import sys
from pathlib import Path

PROJECT2_DIR = Path.cwd().resolve()
REPO_ROOT = PROJECT2_DIR.parents[1]
PROJECTS_DIR = REPO_ROOT / "projects"
for p in (REPO_ROOT, PROJECTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import pandas as pd

from src import config, data

## Load preprocessed dataset

In [7]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)

df = pd.DataFrame(pairs, columns=["text", "label"])
df["label_name"] = df["label"].map(outcome_space.name)
print(len(df), outcome_space)
df.head()

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}
45446 0: admiration, 1: amusement, 2: anger, 3: annoyance, 4: approval, 5: caring, 6: confusion, 7: curiosity, 8: desire, 9: disappointment, 10: disapproval, 11: disgust, 12: embarrassment, 13: excitement, 14: fear, 15: gratitude, 16: grief, 17: joy, 18: love, 19: nervousness, 20: optimism, 21: pride, 22: realization, 23: relief, 24: remorse, 25: sadness, 26: surprise, 27: neutral


,text,label,label_name
0,Fast as [NAME] will carry me. Seriously uptown...,4,approval
1,You blew it. They played you like a fiddle.,27,neutral
2,TL;DR No more Superbowls for [NAME]. Get ready...,9,disappointment
3,So much time saved. Not.,7,curiosity
4,Emotes have a ridiculous amount of effort put ...,27,neutral


## Pretrained embeddings: GloVe Twitter (27B tokens, 1.2M vocab)

The downloaded zip holds four files (25d / 50d / 100d / 200d), but `part2.clslib.PretrainedEmbeddings` only accepts a zip with exactly one file. So we extract the size we want once into `data/cache/` (git-ignored) and load the `.txt` from there.

In [8]:
import zipfile

import part2.clslib

GLOVE_ZIP = PROJECTS_DIR / "src" / "embedders" / "glove.twitter.27B.zip"

  # GloVe Twitter zip with all 4 sizes
CACHE_DIR = PROJECT2_DIR / "data" / "cache"                  # git-ignored


def download_pretrained(dims=200):
    """
    Like the tutorial's download_pretrained, but for GloVe Twitter.
    Extracts glove.twitter.27B.{dims}d.txt from GLOVE_ZIP (skipped if already there).
    Returns (path, dims, skip_first_line) for PretrainedEmbeddings.
    """
    if dims not in (25, 50, 100, 200):
        raise ValueError(f"GloVe Twitter has 25/50/100/200 dims, not {dims}")
    name = f"glove.twitter.27B.{dims}d.txt"
    destination = CACHE_DIR / name
    if not destination.is_file():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(GLOVE_ZIP) as archive:
            archive.extract(name, CACHE_DIR)
    return str(destination), dims, False   # GloVe files have no header line

In [9]:
%%time
embtable_version, embtable_dims, embtable_metadata = download_pretrained(200)
embtable_version, embtable_dims, embtable_metadata

CPU times: user 6.2 s, sys: 750 ms, total: 6.95 s
Wall time: 7.3 s


('/Users/nordin/Documents/UvA/Master/NLP/nlp1-rosa-nordin/projects/project2/data/cache/glove.twitter.27B.200d.txt',
 200,
 False)

In [10]:
%%time
# a few minutes for the 200d file (1.2M words)
trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

CPU times: user 1min 10s, sys: 1.41 s, total: 1min 12s
Wall time: 1min 13s


In [11]:
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}")
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1193515, 200)
Size of vocabulary: 1193515
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:<user>
3:.
4::
5:rt
6:,
7:<repeat>
8:<hashtag>
9:<number>


## MLP classifier (embed-and-pool → MLP → log-softmax)

### Tokenizer

In [12]:
from collections import Counter
from part2 import clslib
import nltk

nltk.download("punkt_tab", quiet=True)

tokenizer = clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
)

# Check how the tokenizer handles Reddit comments
for text, label in pairs[:5]:
    print("Original:", text)
    print("Tokens:  ", tokenizer.tokens(text))
    print()

Original: Fast as [NAME] will carry me. Seriously uptown to downtown via Claiborne or broad isn’t that bad
Tokens:   ['fast', 'as', '[', 'name', ']', 'will', 'carry', 'me', '.', 'seriously', 'uptown', 'to', 'downtown', 'via', 'claiborne', 'or', 'broad', 'isn', '’', 't', 'that', 'bad']

Original: You blew it. They played you like a fiddle.
Tokens:   ['you', 'blew', 'it', '.', 'they', 'played', 'you', 'like', 'a', 'fiddle', '.']

Original: TL;DR No more Superbowls for [NAME]. Get ready for another winning season that ends in disappointment.
Tokens:   ['tl', ';', 'dr', 'no', 'more', 'superbowls', 'for', '[', 'name', ']', '.', 'get', 'ready', 'for', 'another', 'winning', 'season', 'that', 'ends', 'in', 'disappointment', '.']

Original: So much time saved. Not.
Tokens:   ['so', 'much', 'time', 'saved', '.', 'not', '.']

Original: Emotes have a ridiculous amount of effort put into them, which only makes the fact that nobody uses them sadder
Tokens:   ['emotes', 'have', 'a', 'ridiculous', 'am

In [13]:
token_counts = Counter()

for text, label in pairs:
    token_counts.update(tokenizer.tokens(text))

vocab = set(token_counts)

print("Unique tokens:", len(vocab))
print("Total token occurrences:", sum(token_counts.values()))
print("\nMost frequent tokens:")
print(token_counts.most_common(20))

Unique tokens: 29784
Total token occurrences: 697506

Most frequent tokens:
[('.', 37371), ('i', 20475), ('the', 18563), (',', 14413), ('to', 12961), ('a', 12831), ('you', 11402), ('it', 10959), ('that', 9935), ('and', 9227), ('is', 9022), ('!', 8806), ('name', 8489), ('[', 8426), (']', 8424), ('of', 7398), ('’', 6819), ('this', 6591), ('in', 6088), ('for', 5789)]


In [19]:
# Unknown tokens map to the default vocabulary index
unk_idx = trained_emb_model.vocab.default_index

def has_embedding(token):
    return trained_emb_model.vocab.index(token) != unk_idx

# Count how many unique tokens have embeddings
covered_vocab = {
    token for token in vocab
    if has_embedding(token)
}

type_coverage = len(covered_vocab) / len(vocab)

# Frequency-weighted coverage
total_tokens = sum(token_counts.values())
covered_tokens = sum(
    count for token, count in token_counts.items()
    if has_embedding(token)
)

token_coverage = covered_tokens / total_tokens

print(f"Type coverage:  {type_coverage:.2%}")
print(f"Token coverage: {token_coverage:.2%}")

# Most frequent missing tokens
unknown = [
    (token, count)
    for token, count in token_counts.most_common()
    if not has_embedding(token)
]

print("\nMost frequent OOV tokens:")
print(unknown[:30])

Type coverage:  79.47%
Token coverage: 97.97%

Most frequent OOV tokens:
[('...', 2194), ('..', 434), ('....', 348), ('2', 336), ('3', 300), ('1', 200), ('5', 188), ('4', 130), ('10', 129), ('/s', 123), ('100', 110), ('6', 109), ('.....', 97), ('20', 95), ('😂', 75), ('30', 71), ('15', 69), ('7', 66), ('8', 65), ('downvoted', 57), ('2nd', 50), ('9', 44), ('40', 43), ('downvote', 43), ('2019', 40), ("y'all", 38), ('90', 37), ('50', 37), ('25', 37), ('12', 36)]


In [21]:
# semantic similarity between words
import numpy as np

def get_vector(word):
    idx = trained_emb_model.vocab.index(word)

    if idx == trained_emb_model.vocab.default_index:
        return None

    return np.asarray(
        trained_emb_model.vectors[idx],
        dtype=np.float32
    )


def cosine_similarity(word1, word2):
    v1 = get_vector(word1)
    v2 = get_vector(word2)

    if v1 is None or v2 is None:
        return None

    return float(
        np.dot(v1, v2) /
        (np.linalg.norm(v1) * np.linalg.norm(v2))
    )


word_pairs = [
    ("happy", "joyful"),
    ("angry", "furious"),
    ("sad", "depressed"),
    ("grateful", "thankful"),
    ("happy", "angry"),
    ("love", "hate"),
    ("good", "bad"),
]

for w1, w2 in word_pairs:
    similarity = cosine_similarity(w1, w2)

    if similarity is None:
        print(f"{w1:12} {w2:12} OOV")
    else:
        print(f"{w1:12} {w2:12} {similarity:.4f}")

happy        joyful       0.3519
angry        furious      0.3684
sad          depressed    0.7040
grateful     thankful     0.8743
happy        angry        0.4547
love         hate         0.6842
good         bad          0.7984


### Encoder (average of word embeddings)

In [22]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

tokenizer.vocab = trained_emb_model.vocab

pad_idx = trained_emb_model.vocab.index(
    trained_emb_model.pad_token
)

embedding_matrix = torch.as_tensor(
    trained_emb_model.vectors,
    dtype=torch.float32
)

encoder = part2.clslib.EmbedAndPoolEncoder(
    embedding_matrix=embedding_matrix,
    freeze_embeddings=True,
    padding_idx=pad_idx,
)

In [23]:
texts = [text for text, label in pairs[:4]]

sequences = [
    torch.tensor(tokenizer.ids(text), dtype=torch.long)
    for text in texts
]

batch = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=pad_idx,
)

with torch.no_grad():
    representations = encoder(batch)

print("Input shape:", batch.shape)
print("Encoder output shape:", representations.shape)

Input shape: torch.Size([4, 22])
Encoder output shape: torch.Size([4, 200])


### Model (MLP on top of the encoder)

In [ ]:
import torch
import torch.nn as nn


class MLP(nn.Module):
    """One-hidden-layer MLP, copied from part1.smlp.SparseMLP.

    Inputs: feature vectors of size `input_dim` (here: the pooled embedding, e.g. 200d GloVe).
    Process: linear into a hidden layer, ReLU + dropout, linear (num_classes) and log-softmax outputs
        log-softmax, rather than softmax, for numerical stability
    Output: logprobs over `num_classes`.

    Only change vs. part1: the first layer uses a dense matmul (x @ W) instead of
    torch.sparse.mm, since pooled embeddings are dense vectors.
    """

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        """
        input_dim: size of the input feature space (e.g. embedding dimension)
        hidden_dim: width of the hidden layer
        num_classes: number of output labels
        dropout: fraction of hidden units zeroed during training (regularisation)
        """
        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )
        self.bias = nn.Parameter(
            torch.zeros(hidden_dim)
        )

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.output = nn.Sequential(
            nn.Linear(hidden_dim, num_classes),
            nn.LogSoftmax(dim=-1) # for numerical stability, it is common to use log softmax, rather than softmax
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        """Map a batch [batch, input_dim] to class log-probabilities [batch, num_classes]."""
        x = x @ self.weight
        x = x + self.bias

        x = self.activation(x)
        x = self.dropout(x)
        # logprobs
        return self.output(x)


### Cross-validation splits

### Training

### Evaluation